In [2]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb

import pickle
from tqdm import tqdm
import gc
from pathlib import Path

In [3]:
import warnings
import sys
from IPython.core.interactiveshell import InteractiveShell

warnings.filterwarnings("ignore")
sys.path.append("../") # path to the `src`` folder
InteractiveShell.ast_node_interactivity = "all"
tqdm.pandas()

In [4]:
from src.data import DataHelper
from src.data.metrics import map_at_k, hr_at_k, recall_at_k

from src.retrieval.rules import (
    OrderHistory,
    OrderHistoryDecay,
    ItemPair,
    UserGroupTimeHistory,
    UserGroupSaleTrend,
    TimeHistory,
    TimeHistoryDecay,
    SaleTrend,
    OutOfStock,
    
)

from src.retrieval.rules import *


from src.retrieval.collector import RuleCollector

from src.features import full_sale, week_sale, repurchase_ratio, popularity, period_sale

from src.utils import (
    calc_valid_date,
    merge_week_data,
    reduce_mem_usage,
    calc_embd_similarity,
)

In [5]:
data_dir = Path("../data/")
model_dir = Path("../models/")

In [6]:
TRAIN_WEEK_NUM = 4
WEEK_NUM = TRAIN_WEEK_NUM + 2

VERSION_NAME = "Recall 1"
TEST = True # * Set as `False` when do local experiments to save time

In [7]:
import os
if not os.path.exists(data_dir/"interim"/VERSION_NAME):
    os.mkdir(data_dir/"interim"/VERSION_NAME)
if not os.path.exists(data_dir/"processed"/VERSION_NAME):
    os.mkdir(data_dir/"processed"/VERSION_NAME)

# Pepare data: encoding ids and preprocessing

In [8]:
dh = DataHelper(data_dir)

In [8]:
data = dh.preprocess_data(save=True, name="encoded_full") # * run only once, processed data will be saved

Encode Item Sparse Feats: 100%|██████████| 12/12 [00:00<00:00, 30.00it/s]


In [9]:
data = dh.load_data(name="encoded_full")

In [ ]:
user_gender = pd.DataFrame(data['user_gender'])

In [25]:
user_gender.head(5)

,customer_id,0,1,2
0,1,1,4,16
1,2,1,3,82
2,3,0,4,14
3,4,0,0,2
4,5,0,0,13


In [21]:
item = pd.DataFrame(data["item"])

In [23]:
item.count()

article_id                    105542
product_code                  105542
product_type_no               105542
product_group_name            105542
graphical_appearance_no       105542
colour_group_code             105542
perceived_colour_value_id     105542
perceived_colour_master_id    105542
department_no                 105542
index_code                    105542
index_group_no                105542
section_no                    105542
garment_group_no              105542
article_gender                105542
season_type                   105542
dtype: int64

In [19]:
def assign_gender_to_user(user_gender: pd.DataFrame, min_slots: int, proportion: float) -> pd.DataFrame:
    counts = user_gender[['0','1','2']].to_numpy()

    # max index & value
    first_idx = counts.argmax(axis=1)
    first_val = counts[np.arange(len(counts)), first_idx]

    # second max value
    masked = counts.copy()
    masked[np.arange(len(counts)), first_idx] = -1
    second_val = masked.max(axis=1)

    gender = np.zeros(len(counts), dtype=int)

    cond = (first_val > min_slots) & (
        (second_val == 0) |
        (first_val / np.maximum(second_val, 1e-9) >= proportion)
    )

    gender[cond] = first_idx[cond]

    return pd.DataFrame({
        "customer_id": user_gender["customer_id"].values,
        "gender": gender
    })

In [ ]:
def gender_filter(transactions: pd.DataFrame, 
                  user_gender: pd.DataFrame, 
                  item: pd.DataFrame, 
                  gender_min_slots: int, 
                  gender_proportion: float,
                  prob: float) -> pd.DataFrame:

    user_gender = assign_gender_to_user(user_gender=user_gender, 
                                        min_slots=gender_min_slots,
                                        proportion=gender_proportion)
    transactions = transactions[['customer_id', 'article_id']]
    item_gender = item[['article_id', 'article_gender']]

    user_item = user_gender.merge(transactions, on='customer_id')
    user_item = user_item.merge(item_gender, on='article_id')

    # điều kiện 1: user gender = 0  -> giữ hết
    cond_user_unknown = user_item["gender"] == 0

    # điều kiện 2: gender khớp
    cond_match = user_item["gender"] == user_item["article_gender"]

    # điều kiện 3: article gender = 0 -> giữ
    cond_article_unisex = user_item["article_gender"] == 0

    # điều kiện 4: mismatch -> lấy theo xác suất prob
    cond_mismatch = (
        (user_item["gender"] != 0)
        & (user_item["article_gender"] != 0)
        & (user_item["gender"] != user_item["article_gender"])
    )

    random_mask = np.random.rand(len(user_item)) < prob

    cond_keep = (
        cond_user_unknown
        | cond_match
        | cond_article_unisex
        | (cond_mismatch & random_mask)
    )

    return user_item.loc[cond_keep, ["customer_id", "article_id"]]